# Trajectory prediction training (scene-level + BiGRU)

Uses **multi-vehicle scene dataset** and **BiGRU temporal encoder**. Same setup as `explore_argoverse.ipynb`: clone repo, switch branch, download dataset.

## 1. Setup (Colab: clone repo, project path, install deps)

In [ ]:
# Clone project (Colab: run this first). If repo exists, clean cache and pull branch.
import os
import sys

REPO_URL = "https://github.com/PulockDas/Implement-STAST-System.git"  # set your repo URL
BRANCH = "master"  # branch to checkout
CLONE_DIR = "/content/Implement-STAST-System"  # Colab; use os.getcwd() for local

ip = get_ipython()
if os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    ip.run_line_magic("cd", CLONE_DIR)
    ip.system("git fetch origin")
    ip.system("git clean -fd")
    ip.system(f"git checkout {BRANCH}")
    ip.system(f"git pull origin {BRANCH}")
    print(f"Updated existing repo at {CLONE_DIR} (branch {BRANCH})")
else:
    parent = os.path.dirname(CLONE_DIR) or "."
    os.makedirs(parent, exist_ok=True)
    ip.run_line_magic("cd", parent)
    ip.system(f"git clone --branch {BRANCH} {REPO_URL} {os.path.basename(CLONE_DIR)}")
    print(f"Cloned repo to {CLONE_DIR} (branch {BRANCH})")

PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
ip.run_line_magic("cd", PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# If you didn't run the clone cell (e.g. local): set project root to parent of notebooks/.
import sys
import os

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# Install dependencies (skip if already installed, e.g. in Colab)
!pip install -q tqdm

## 2. Download dataset with kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("narendarmallireddy/argoverse1-motion-dataset")
print("Dataset path:", path)

In [ ]:
from pathlib import Path

data_path = Path(path)
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
if not csv_files:
    raise FileNotFoundError("No CSV files found under dataset path.")

## 3. Scene-level dataset and DataLoader

**How much data feeds the BiGRU:** We use **max_scenes** scenarios (one CSV = one scene = one training sample). Default below: **500** scenes. Each scene has multiple vehicles (past_traj shape N ≤ max_vehicles). So total training samples = number of valid scenes built (≤ max_scenes).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from datasets.argoverse_dataset import ArgoverseSceneDataset
from utils.config import PAST_STEPS, FUTURE_STEPS

# Number of scenarios (CSV files) to use for training/validation
max_scenes = 5000  # adjust down (e.g. 1000) if Colab is slow
scene_csvs = csv_files[:max_scenes]

full_dataset = ArgoverseSceneDataset(
    scene_csvs,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    max_vehicles=20,
    distance_threshold=50.0,
)

# Train / val split
val_fraction = 0.1
val_size = max(1, int(len(full_dataset) * val_fraction))
train_size = max(1, len(full_dataset) - val_size)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Total scenes:", len(full_dataset))
print("Train scenes:", len(train_dataset))
print("Val scenes:", len(val_dataset))
print("Device:", device)

## 3b. Verify dataset output (normalized coordinates)

Shapes should be `(B, N, 20, 2)` and `(B, 30, 2)`. Target vehicle's last past point should be near **(0, 0)**.

In [ ]:
batch = next(iter(train_loader))
print("past_traj shape:", batch["past_traj"].shape)
print("future_target shape:", batch["future_target"].shape)

# Verify last past point of target (index 0) is near (0, 0)
past_target = batch["past_traj"][0, 0]  # first sample, target vehicle
last_past = past_target[-1]
print("Last past point (target vehicle, sample 0):", last_past.tolist(), "-> should be ~ (0, 0)")

from visualization import plot_trajectory

past_np = batch["past_traj"][0, 0].numpy()
gt_np = batch["future_target"][0].numpy()
# Dummy pred (gt copy) just to show past + future; past should end at (0,0)
plot_trajectory(past_np, gt_np, gt_np.copy(), title="Verify normalization (past ends at origin)")

## 4. BiGRU temporal encoder — shape test

In [ ]:
from models.temporal_encoder import BiGRUTrajectoryEncoder

batch = next(iter(train_loader))
past = batch["past_traj"]  # (B, N, 20, 2)

model = BiGRUTrajectoryEncoder()
vehicle_features = model.encoder(past)
print("vehicle_features.shape:", tuple(vehicle_features.shape))
print("expected: (B, N, 128)")

pred_traj = model(past)
print("pred_traj.shape:", tuple(pred_traj.shape))
print("expected: (B, 30, 2)")

## 5. Train BiGRU (MSE on future_target)

In [ ]:
from metrics.trajectory_metrics import ade, fde

model = BiGRUTrajectoryEncoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

metrics_log = []
EPOCHS = 10
for epoch in range(EPOCHS):
    # ------------------ train ------------------
    model.train()
    total_loss = 0.0
    total_ade = 0.0
    total_fde = 0.0
    n_batches = 0
    for batch in train_loader:
        past_scene = batch["past_traj"].to(device)
        future_target = batch["future_target"].to(device)
        optimizer.zero_grad()
        pred = model(past_scene)
        loss = criterion(pred, future_target)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            total_loss += loss.item()
            total_ade += ade(pred, future_target).item()
            total_fde += fde(pred, future_target).item()
            n_batches += 1
    train_mse = total_loss / max(n_batches, 1)
    train_ade = total_ade / max(n_batches, 1)
    train_fde = total_fde / max(n_batches, 1)

    # ------------------ validation ------------------
    model.eval()
    val_loss = 0.0
    val_ade = 0.0
    val_fde = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            past_scene = batch["past_traj"].to(device)
            future_target = batch["future_target"].to(device)
            pred = model(past_scene)
            loss = criterion(pred, future_target)
            val_loss += loss.item()
            val_ade += ade(pred, future_target).item()
            val_fde += fde(pred, future_target).item()
            val_batches += 1
    val_mse = val_loss / max(val_batches, 1)
    val_ade = val_ade / max(val_batches, 1)
    val_fde = val_fde / max(val_batches, 1)

    print(
        f"Epoch {epoch+1}/{EPOCHS}  "
        f"train MSE: {train_mse:.6f}  ADE: {train_ade:.4f}  FDE: {train_fde:.4f}  "
        f"val MSE: {val_mse:.6f}  ADE: {val_ade:.4f}  FDE: {val_fde:.4f}"
    )
    metrics_log.append({
        "epoch": epoch + 1,
        "train_ade": train_ade,
        "train_fde": train_fde,
        "val_ade": val_ade,
        "val_fde": val_fde,
    })

print("Training finished.")

In [ ]:
# 6. Trajectory visualization (helper in visualization/plot_trajectories.py)
from visualization import plot_trajectory

model.eval()
val_batch = next(iter(val_loader))
with torch.no_grad():
    pred_val = model(val_batch["past_traj"].to(device)).cpu().numpy()

past_scene = val_batch["past_traj"].numpy()        # (B, N, 20, 2)
future_target = val_batch["future_target"].numpy()  # (B, 30, 2)

b = 0  # sample index within batch
past_target = past_scene[b, 0]       # (20, 2)
gt_future = future_target[b]         # (30, 2)
pred_future = pred_val[b]            # (30, 2)

plot_trajectory(past_target, gt_future, pred_future)

In [ ]:
# 7. Save checkpoints and training metrics
import os
import json

os.makedirs("checkpoints", exist_ok=True)

torch.save(model.state_dict(), "checkpoints/bigru_temporal_model.pth")
print("Saved weights: checkpoints/bigru_temporal_model.pth")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "checkpoints/bigru_temporal_checkpoint.pth")
print("Saved full checkpoint: checkpoints/bigru_temporal_checkpoint.pth")

with open("checkpoints/bigru_training_metrics.json", "w") as f:
    json.dump(metrics_log, f, indent=2)
print("Saved metrics: checkpoints/bigru_training_metrics.json")

# Quick inference test
batch = next(iter(val_loader))
with torch.no_grad():
    pred = model(batch["past_traj"].to(device))
print("Inference pred shape:", tuple(pred.shape), "-> expected (B, 30, 2)")

## 8. BiGRU + Graph (CGConv) — Step 5: Vehicle Interaction

Pipeline: past_traj → BiGRU encoder → vehicle_features → CGConv interaction → decoder → pred_traj.

In [ ]:
# BiGRU + Graph model — shape test
from models.temporal_encoder import BiGRUGraphTrajectoryEncoder
from models.graph_layers import build_adjacency_matrix

batch = next(iter(train_loader))
past = batch["past_traj"]  # (B, N, 20, 2)
vehicle_mask = batch["vehicle_mask"]  # (B, N)

adj = build_adjacency_matrix(past, vehicle_mask)
print("adjacency shape:", tuple(adj.shape), "expected: (B, N, N)")

model_graph = BiGRUGraphTrajectoryEncoder()
pred_traj = model_graph(past, vehicle_mask)
print("pred_traj.shape:", tuple(pred_traj.shape), "expected: (B, 30, 2)")

In [ ]:
# Train BiGRU + Graph (MSE, ADE, FDE)
from metrics.trajectory_metrics import ade, fde

model_graph = BiGRUGraphTrajectoryEncoder().to(device)
criterion = nn.MSELoss()
optimizer_graph = torch.optim.Adam(model_graph.parameters(), lr=1e-3)

metrics_log_graph = []
EPOCHS = 10
for epoch in range(EPOCHS):
    model_graph.train()
    total_loss = 0.0
    total_ade = 0.0
    total_fde = 0.0
    n_batches = 0
    for batch in train_loader:
        past_scene = batch["past_traj"].to(device)
        future_target = batch["future_target"].to(device)
        vehicle_mask = batch["vehicle_mask"].to(device)
        optimizer_graph.zero_grad()
        pred = model_graph(past_scene, vehicle_mask)
        loss = criterion(pred, future_target)
        loss.backward()
        optimizer_graph.step()
        with torch.no_grad():
            total_loss += loss.item()
            total_ade += ade(pred, future_target).item()
            total_fde += fde(pred, future_target).item()
            n_batches += 1
    train_mse = total_loss / max(n_batches, 1)
    train_ade = total_ade / max(n_batches, 1)
    train_fde = total_fde / max(n_batches, 1)

    model_graph.eval()
    val_loss = 0.0
    val_ade = 0.0
    val_fde = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            past_scene = batch["past_traj"].to(device)
            future_target = batch["future_target"].to(device)
            vehicle_mask = batch["vehicle_mask"].to(device)
            pred = model_graph(past_scene, vehicle_mask)
            loss = criterion(pred, future_target)
            val_loss += loss.item()
            val_ade += ade(pred, future_target).item()
            val_fde += fde(pred, future_target).item()
            val_batches += 1
    val_mse = val_loss / max(val_batches, 1)
    val_ade = val_ade / max(val_batches, 1)
    val_fde = val_fde / max(val_batches, 1)

    print(
        f"Epoch {epoch+1}/{EPOCHS}  "
        f"train MSE: {train_mse:.6f}  ADE: {train_ade:.4f}  FDE: {train_fde:.4f}  "
        f"val MSE: {val_mse:.6f}  ADE: {val_ade:.4f}  FDE: {val_fde:.4f}"
    )
    metrics_log_graph.append({
        "epoch": epoch + 1,
        "train_ade": train_ade,
        "train_fde": train_fde,
        "val_ade": val_ade,
        "val_fde": val_fde,
    })

print("BiGRU+Graph training finished.")

In [ ]:
# BiGRU+Graph trajectory visualization
model_graph.eval()
val_batch = next(iter(val_loader))
with torch.no_grad():
    pred_val = model_graph(
        val_batch["past_traj"].to(device),
        val_batch["vehicle_mask"].to(device),
    ).cpu().numpy()

past_scene = val_batch["past_traj"].numpy()
future_target = val_batch["future_target"].numpy()
b = 0
plot_trajectory(past_scene[b, 0], future_target[b], pred_val[b], title="BiGRU+Graph prediction")

In [ ]:
# Save BiGRU+Graph model checkpoint
torch.save(model_graph.state_dict(), "checkpoints/bigru_graph_model.pth")
print("Saved weights: checkpoints/bigru_graph_model.pth")

torch.save({
    "model_state_dict": model_graph.state_dict(),
    "optimizer_state_dict": optimizer_graph.state_dict(),
}, "checkpoints/bigru_graph_checkpoint.pth")
print("Saved full checkpoint: checkpoints/bigru_graph_checkpoint.pth")

with open("checkpoints/bigru_graph_training_metrics.json", "w") as f:
    json.dump(metrics_log_graph, f, indent=2)
print("Saved metrics: checkpoints/bigru_graph_training_metrics.json")